In [0]:

# Task 1: Data Center IoT Ingestion
# FastAPI -> Raw CSV -> Bronze Delta
# Shared process_date from Databricks Job

import time
import requests
from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

from pyspark.sql.functions import to_date, lit

from pipeline_logger import logger, save_logs_to_volume


try:
    logger.info("===== DATA CENTER INGESTION STARTED =====")

    base = "/Volumes/workspace/iot_db/data"

    # STEP 1: Define 15 columns
    # Bronze mein raw values StringType mein rahengi.
    # Silver mein cleaning aur type conversion hogi.

    columns = [
        "reading_id",
        "sensor_id",
        "data_center_id",
        "region",
        "city",
        "rack_id",
        "reading_timestamp",
        "temperature_c",
        "humidity_percent",
        "cooling_status",
        "power_consumption_kw",
        "sensor_status",
        "alert_level",
        "process_date",
        "ingestion_timestamp"
    ]

    schema = StructType([
        StructField(name, StringType(), True)
        for name in columns
    ])

    # STEP 2: Get shared process date from Job
    # Job parameter:
    # {{tasks.00_prepare_date.values.process_date}}

    dbutils.widgets.text(
        "process_date",
        "",
        "Process Date"
    )

    process_date = dbutils.widgets.get(
        "process_date"
    ).strip()

    if not process_date:
        raise ValueError(
            "process_date is required from prepare_date task"
        )

    # Validate date format
    datetime.strptime(process_date, "%Y-%m-%d")

    # STEP 3: Get total rows from Job
    # Job parameter: rows = 1500000

    dbutils.widgets.text(
        "rows",
        "1500000",
        "Total Rows"
    )

    total_rows = int(dbutils.widgets.get("rows"))

    if total_rows <= 0:
        raise ValueError(
            "rows must be greater than zero"
        )

    chunk_size = 100000

    logger.info(
        f"Date: {process_date}, Rows: {total_rows}"
    )

    print("Process Date:", process_date)
    print("Requested Rows:", total_rows)

    # STEP 4: Prepare Raw folder
    raw_folder = (
        f"{base}/dc_raw/process_date={process_date}"
    )

    dbutils.fs.rm(raw_folder, True)
    dbutils.fs.mkdirs(raw_folder)

    # STEP 5: Download CSV from FastAPI
    api_url = ( "YOUR_IOT_API_URL_HERE"
    
    
    )

    chunks = (
        total_rows + chunk_size - 1
    ) // chunk_size

    for i in range(chunks):

        rows = min(
            chunk_size,
            total_rows - i * chunk_size
        )

        for attempt in range(1, 4):

            try:
                response = requests.get(
                    api_url,
                    params={
                        "process_date": process_date,
                        "rows": rows
                    },
                    timeout=300
                )

                response.raise_for_status()
                break

            except Exception as e:

                logger.warning(
                    f"Chunk {i + 1}, attempt {attempt}: {e}"
                )

                if attempt == 3:
                    raise

                time.sleep(20)

        chunk_file = (
            f"{raw_folder}/part_{i + 1:03d}.csv"
        )

        with open(chunk_file, "wb") as file:
            file.write(response.content)

        logger.info(
            f"Chunk {i + 1}/{chunks} downloaded"
        )

    # STEP 6: Read Raw CSV
    df = (
        spark.read
        .option("header", True)
        .schema(schema)
        .csv(raw_folder)
    )

    records_read = df.count()

    # Generator adds 1% duplicate rows per chunk
    expected_rows = total_rows + sum(
        min(
            chunk_size,
            total_rows - i * chunk_size
        ) // 100
        for i in range(chunks)
    )

    if records_read != expected_rows:
        raise ValueError(
            f"Expected {expected_rows}, got {records_read}"
        )

    logger.info(
        f"Bronze input records: {records_read}"
    )

    # STEP 7: Set process_date for partitioning
    df = df.withColumn(
        "process_date",
        to_date(lit(process_date))
    )

    # STEP 8: Save Bronze Delta
    bronze_path = f"{base}/dc_bronze"

    # Sirf selected date ka partition overwrite hoga.
    # Doosri dates ka data safe rahega.

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"process_date = '{process_date}'"
        )
        .partitionBy("process_date")
        .save(bronze_path)
    )

    logger.info(
        f"Bronze ingestion successful: {records_read} records"
    )

    print("DATA CENTER BRONZE INGESTION SUCCESSFUL")
    print("Process Date:", process_date)
    print("Total Records:", records_read)
    print("Bronze Path:", bronze_path)

except Exception:
    logger.exception("Data Center ingestion failed")
    raise

finally:
    save_logs_to_volume(dbutils)
